# Transfer Learning for SteelBlast Image Classification

Using pre-trained neural networks (VGG16, InceptionV3, ResNet50) to classify steel surfaces as "good" or "not-good" for shot-blasting quality assessment.

## 1. Import Required Libraries

In [ ]:
from __future__ import annotations

import os
import json
import tempfile
from pathlib import Path
from dataclasses import dataclass

# Set matplotlib backend before importing
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))
# Enable Metal GPU allocator for Apple Silicon
os.environ.setdefault("TF_GPU_ALLOCATOR", "metal")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import VGG16, InceptionV3, ResNet50
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Enabled memory growth for Apple GPU")
    except RuntimeError as e:
        print("Could not set memory growth:", e)


## 2. Configuration and Dataset Loading

In [ ]:
@dataclass(frozen=True)
class AppConfig:
    dataset_dir: Path = Path("doi-10.34894-ekznn0(1)/SteelBlastQC")
    # Model choice: 'VGG16', 'InceptionV3', or 'ResNet50'
    model_choice: str = 'VGG16'
    image_size: tuple = (224, 224)  # VGG16 and ResNet50 use 224x224, InceptionV3 uses 299x299
    batch_size: int = 32
    epochs: int = 50
    learning_rate: float = 0.001
    validation_split: float = 0.2
    quick_limit: int | None = None

LABELS = {"good": 0, "not-good": 1}
CLASS_NAMES = ["good", "not-good"]

app_config = AppConfig()

# Adjust image size based on model choice
if app_config.model_choice == 'InceptionV3':
    app_config = AppConfig(image_size=(299, 299), model_choice='InceptionV3')
    
print(f"Model: {app_config.model_choice}")
print(f"Image size: {app_config.image_size}")
print(f"Dataset directory: {app_config.dataset_dir}")

## 3. Load and Explore the Dataset

In [ ]:
def load_split(dataset_dir: Path, split: str) -> tuple[list[Path], np.ndarray]:
    """Load image paths and labels for train or test split."""
    image_paths: list[Path] = []
    labels: list[int] = []

    for class_name, label in LABELS.items():
        class_dir = dataset_dir / split / class_name
        if not class_dir.exists():
            raise FileNotFoundError(f"Missing expected folder: {class_dir}")

        paths = sorted(class_dir.glob("*.png"))
        image_paths.extend(paths)
        labels.extend([label] * len(paths))

    if not image_paths:
        raise FileNotFoundError(f"No .png images found in {dataset_dir / split}")

    return image_paths, np.asarray(labels, dtype=np.int64)

# Load dataset splits
train_paths, y_train = load_split(app_config.dataset_dir, "train")
test_paths, y_test = load_split(app_config.dataset_dir, "test")

print(f"Train set: {len(train_paths)} images")
print(f"  - Good: {(y_train == 0).sum()}")
print(f"  - Not-good: {(y_train == 1).sum()}")
print(f"\nTest set: {len(test_paths)} images")
print(f"  - Good: {(y_test == 0).sum()}")
print(f"  - Not-good: {(y_test == 1).sum()}")

# Split train into train and validation
train_paths, val_paths, y_train, y_val = train_test_split(
    train_paths, y_train, test_size=app_config.validation_split, 
    random_state=42, stratify=y_train
)

print(f"\nAfter train/val split:")
print(f"Train: {len(train_paths)}, Validation: {len(val_paths)}, Test: {len(test_paths)}")

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
fig.suptitle("Sample Images: Good vs Not-Good", fontsize=14, fontweight='bold')

# Get sample indices
good_indices = np.where(y_train == 0)[0][:4]
notgood_indices = np.where(y_train == 1)[0][:4]

for i, idx in enumerate(good_indices):
    img = load_img(train_paths[idx])
    axes[0, i].imshow(img)
    axes[0, i].set_title("Good")
    axes[0, i].axis("off")

for i, idx in enumerate(notgood_indices):
    img = load_img(train_paths[idx])
    axes[1, i].imshow(img)
    axes[1, i].set_title("Not-Good")
    axes[1, i].axis("off")

plt.tight_layout()
plt.savefig("dataset_samples.png", dpi=100, bbox_inches='tight')
plt.show()
print("Sample images saved to 'dataset_samples.png'")

## 4. Preprocess Images for Transfer Learning

In [ ]:
def load_and_preprocess_image(image_path: Path, target_size: tuple, preprocess_fn):
    """Load and preprocess a single image."""
    img = load_img(image_path, target_size=target_size)
    img_array = img_to_array(img)
    img_array = preprocess_fn(img_array)
    return img_array

def create_data_arrays(image_paths: list[Path], labels: np.ndarray, 
                       target_size: tuple, preprocess_fn) -> tuple:
    """Create numpy arrays of all images and labels."""
    images = []
    for path in image_paths:
        img = load_and_preprocess_image(path, target_size, preprocess_fn)
        images.append(img)
    return np.array(images), labels

# Select preprocessing function based on model choice
if app_config.model_choice == 'VGG16':
    preprocess_fn = vgg16_preprocess
elif app_config.model_choice == 'InceptionV3':
    preprocess_fn = inception_preprocess
else:  # ResNet50
    preprocess_fn = resnet_preprocess

print(f"Loading and preprocessing images with {app_config.model_choice}...")
print(f"Target size: {app_config.image_size}")

# Load all image data
X_train, y_train = create_data_arrays(train_paths, y_train, app_config.image_size, preprocess_fn)
X_val, y_val = create_data_arrays(val_paths, y_val, app_config.image_size, preprocess_fn)
X_test, y_test = create_data_arrays(test_paths, y_test, app_config.image_size, preprocess_fn)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")

# Convert labels to one-hot encoding for categorical crossentropy
y_train_cat = keras.utils.to_categorical(y_train, num_classes=2)
y_val_cat = keras.utils.to_categorical(y_val, num_classes=2)
y_test_cat = keras.utils.to_categorical(y_test, num_classes=2)

print(f"Training labels shape: {y_train_cat.shape}")

## 5. Load Pre-trained Model (VGG16, InceptionV3, or ResNet50)

In [ ]:
# Load pre-trained model with ImageNet weights
print(f"Loading {app_config.model_choice} with ImageNet weights...")

if app_config.model_choice == 'VGG16':
    base_model = VGG16(
        input_shape=(*app_config.image_size, 3),
        include_top=False,
        weights='imagenet'
    )
elif app_config.model_choice == 'InceptionV3':
    base_model = InceptionV3(
        input_shape=(*app_config.image_size, 3),
        include_top=False,
        weights='imagenet'
    )
else:  # ResNet50
    base_model = ResNet50(
        input_shape=(*app_config.image_size, 3),
        include_top=False,
        weights='imagenet'
    )

# Freeze base model layers to preserve learned features
base_model.trainable = False
print(f"Base model loaded with {len(base_model.layers)} layers")
print(f"Trainable parameters in base model: {base_model.count_params():,}")

## 6. Build the Classification Model

In [ ]:
# Build custom classification model on top of base model
print("Building classification model...")

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu', name='dense_1'),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu', name='dense_2'),
    layers.Dropout(0.3),
    # Output layer for binary classification
    layers.Dense(2, activation='softmax', name='output')
])

print(f"Model architecture:")
model.summary()

## 7. Compile the Model

In [ ]:
# Compile the model
optimizer = keras.optimizers.Adam(learning_rate=app_config.learning_rate)

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully")
print(f"Optimizer: Adam (lr={app_config.learning_rate})")
print(f"Loss: categorical_crossentropy")
print(f"Metrics: accuracy")

## 8. Train the Model

In [ ]:
# Define callbacks for training
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        f'steelblast_{app_config.model_choice.lower()}_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

# Create tf.data datasets for faster training throughput
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train_cat))
train_dataset = train_dataset.cache()
train_dataset = train_dataset.shuffle(buffer_size=len(X_train), reshuffle_each_iteration=True)
train_dataset = train_dataset.batch(app_config.batch_size)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val_cat))
val_dataset = val_dataset.cache()
val_dataset = val_dataset.batch(app_config.batch_size)
val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

# Train the model
print(f"Training {app_config.model_choice} model...")
print(f"Epochs: {app_config.epochs}, Batch size: {app_config.batch_size}")

history = model.fit(
    train_dataset,
    epochs=app_config.epochs,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining completed!")

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title(f'{app_config.model_choice} - Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title(f'{app_config.model_choice} - Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f'steelblast_{app_config.model_choice.lower()}_training_history.png', dpi=100, bbox_inches='tight')
plt.show()
print("Training history plot saved")

## 9. Evaluate Model Performance

In [ ]:
# Evaluate on test set
print("Evaluating model on test set...")
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\nTest Results:")
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_accuracy:.4f}")

# Get predictions
y_pred_proba = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)

# Calculate detailed metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"\nDetailed Metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

# Classification report
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, 
            cbar=True, ax=ax)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
ax.set_title(f'{app_config.model_choice} - Confusion Matrix')
plt.tight_layout()
plt.savefig(f'steelblast_{app_config.model_choice.lower()}_confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Confusion Matrix:\n{cm}")

## 10. Make Predictions on Test Images

In [ ]:
# Visualize predictions on test images
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle(f'{app_config.model_choice} - Predictions on Test Images', fontsize=14, fontweight='bold')

# Random sample of test images
sample_indices = np.random.choice(len(test_paths), size=12, replace=False)

for idx, ax in enumerate(axes.flat):
    test_idx = sample_indices[idx]
    
    # Load original image for display
    img_display = load_img(test_paths[test_idx])
    ax.imshow(img_display)
    
    # Get prediction
    pred_class = y_pred[test_idx]
    pred_prob = y_pred_proba[test_idx]
    true_class = y_test[test_idx]
    
    # Color based on correctness
    color = 'green' if pred_class == true_class else 'red'
    
    title = f"True: {CLASS_NAMES[true_class]}\n"
    title += f"Pred: {CLASS_NAMES[pred_class]} ({pred_prob[pred_class]:.2%})"
    
    ax.set_title(title, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'steelblast_{app_config.model_choice.lower()}_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

print("Prediction visualization saved")

## 11. Save Model and Metrics

In [ ]:
# Save the trained model
model_path = f'steelblast_{app_config.model_choice.lower()}.h5'
model.save(model_path)
print(f"Model saved to {model_path}")

# Save metrics to JSON
metrics = {
    "model": app_config.model_choice,
    "image_size": app_config.image_size,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "dataset_info": {
        "train_samples": len(train_paths),
        "val_samples": len(val_paths),
        "test_samples": len(test_paths),
        "classes": CLASS_NAMES
    },
    "confusion_matrix": cm.tolist(),
    "training_epochs": len(history.history['loss']),
    "batch_size": app_config.batch_size
}

metrics_path = f'steelblast_{app_config.model_choice.lower()}_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=4)
print(f"Metrics saved to {metrics_path}")

# Display final summary
print("\n" + "="*60)
print(f"FINAL SUMMARY - {app_config.model_choice}")
print("="*60)
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test Loss:      {test_loss:.4f}")
print(f"Precision:      {precision:.4f}")
print(f"Recall:         {recall:.4f}")
print(f"F1-Score:       {f1:.4f}")
print("="*60)

## 12. Model Comparison and Fine-tuning Tips

### Switching Between Models
To use a different pre-trained model, modify the `model_choice` parameter in the configuration cell:
- **VGG16**: Simple, interpretable, good baseline (224×224)
- **InceptionV3**: Efficient multi-scale feature extraction (299×299)
- **ResNet50**: Better performance with residual connections (224×224)

### Fine-tuning Strategies
1. **Unfreeze Later Layers**: After initial training, unfreeze the last few layers of the base model and fine-tune with a lower learning rate
2. **Data Augmentation**: Add rotation, zoom, and horizontal flip during training for better generalization
3. **Adjust Dropout Rates**: Increase dropout if overfitting occurs
4. **Learning Rate Tuning**: Use the ReduceLROnPlateau callback to adaptively reduce learning rate

### Expected Performance
- VGG16: ~85-90% accuracy (lower complexity)
- InceptionV3: ~88-92% accuracy (balanced)
- ResNet50: ~90-93% accuracy (best overall)